# Basketball Detection Model Training - Google Colab

This notebook is designed to train YOLO11 detection models using Google Colab's GPU resources.

## Setup Instructions

1. **Mount Google Drive** (optional, for saving models)
2. **Clone Repository** - Get the latest code
3. **Install Dependencies** - Install required packages
4. **Configure API Keys** - Set up Roboflow credentials
5. **Download Dataset** - Download dataset from Roboflow (YOLOv11 format)
6. **Train Model** - Train YOLO11s model
7. **Evaluate Results** - Analyze training metrics
8. **Save Model** - Save best model weights

---

**Note**: 
- Make sure to set your Roboflow API key in the configuration section below
- The dataset class count is automatically read from `data.yaml` - no manual changes needed if you've updated classes in Roboflow

## 1. Setup and Installation

In [ ]:
# Mount Google Drive (optional - for saving models to Drive)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository
!git clone https://github.com/TomsFridrihsons/Basketball_shot_analytics_CV.git
%cd Basketball_shot_analytics_CV

# Checkout colab-training branch (if using separate branch)
# !git checkout colab-training

In [ ]:
# Install required packages
!pip install ultralytics roboflow pyyaml

## 2. Configuration - API Keys and Settings

**⚠️ IMPORTANT**: Set your API key below. For security, use Colab's secrets manager or environment variables.

In [ ]:
# ============================================
# CONFIGURATION - Update these values
# ============================================

import os
from google.colab import userdata

# ============================================
# API KEY SETUP - REQUIRED
# ============================================
# Method 1 (RECOMMENDED): Use Colab Secrets
#   1. Click the 🔑 key icon in Colab sidebar
#   2. Click "Add new secret"
#   3. Name: ROBOFLOW_API_KEY
#   4. Value: Your Roboflow API key
#   5. The code below will automatically use it

try:
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    # Method 2: Set API key directly (ONLY for personal notebooks)
    # ⚠️ WARNING: Remove this before sharing your notebook!
    ROBOFLOW_API_KEY = input("Enter your Roboflow API key: ").strip()
    if not ROBOFLOW_API_KEY:
        raise ValueError("❌ API key is required! Please set it using Colab secrets or enter it above.")

# Roboflow project configuration
ROBOFLOW_WORKSPACE = "basketball-dataset"
ROBOFLOW_PROJECT = "hooper_dataset_3006_2235"
ROBOFLOW_VERSION = 3

# Training configuration
MODEL_SIZE = "s"  # Options: n (nano), s (small), m (medium), l (large), x (xlarge)
# Using YOLO11s (small) model
YOLO_VERSION = "11"  # YOLO version: 8, 11, etc.
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 16
PROJECT_NAME = "basketball-detection-colab-yolo11s"

print(f"\n✅ Configuration loaded")
print(f"   Model: YOLO{YOLO_VERSION}{MODEL_SIZE} (YOLO11s)")
print(f"   Epochs: {EPOCHS}")
print(f"   Image Size: {IMG_SIZE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Workspace: {ROBOFLOW_WORKSPACE}")
print(f"   Project: {ROBOFLOW_PROJECT}")
print(f"   Version: {ROBOFLOW_VERSION}")
print(f"\n📝 Note: Dataset class count will be read automatically from data.yaml")

## 3. Download Dataset from Roboflow

In [ ]:
# Download dataset from Roboflow
from roboflow import Roboflow
import os

# Initialize Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Get project and version
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)

# Download dataset in YOLOv11 format
# Note: If you've updated classes in Roboflow, the data.yaml will reflect the new class count
dataset = version.download("yolov11")

print(f"✅ Dataset downloaded to: {dataset.location}")
print(f"   Dataset path: {dataset.location}")

# Set dataset path for training
DATASET_PATH = dataset.location
DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")

print(f"\n📁 Dataset configuration file: {DATA_YAML}")
print(f"   Verify data.yaml exists: {os.path.exists(DATA_YAML)}")

## 4. Verify Dataset Structure

In [ ]:
# Check dataset structure
import yaml

# Read data.yaml
with open(DATA_YAML, 'r') as f:
    dataset_config = yaml.safe_load(f)

print("📊 Dataset Configuration:")
num_classes = dataset_config.get('nc', 'N/A')
class_names = dataset_config.get('names', 'N/A')
print(f"   Number of Classes: {num_classes}")
print(f"   Class Names: {class_names}")
print(f"   Train: {dataset_config.get('train', 'N/A')}")
print(f"   Val: {dataset_config.get('val', 'N/A')}")

# Verify class count matches expectations
if isinstance(num_classes, int):
    print(f"\n✅ Dataset has {num_classes} classes (automatically detected from Roboflow)")
    if isinstance(class_names, dict):
        print(f"   Classes: {list(class_names.values())}")
else:
    print(f"\n⚠️ Could not determine class count from data.yaml")

# Check if directories exist
train_path = os.path.join(DATASET_PATH, dataset_config.get('train', ''))
val_path = os.path.join(DATASET_PATH, dataset_config.get('val', ''))

if os.path.exists(train_path):
    train_images = len([f for f in os.listdir(os.path.join(train_path, 'images')) if f.endswith(('.jpg', '.png'))])
    print(f"\n✅ Training images: {train_images}")
else:
    print(f"\n⚠️ Training path not found: {train_path}")

if os.path.exists(val_path):
    val_images = len([f for f in os.listdir(os.path.join(val_path, 'images')) if f.endswith(('.jpg', '.png'))])
    print(f"✅ Validation images: {val_images}")
else:
    print(f"⚠️ Validation path not found: {val_path}")

## 5. Train YOLO Model

In [ ]:
# Train YOLO11 model
from ultralytics import YOLO
import torch

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")

# Load YOLO11s model
# Note: YOLO11 models use the same naming convention as YOLOv8
model = YOLO(f'yolov11{MODEL_SIZE}.pt')
print(f"\n✅ Loaded model: yolov11{MODEL_SIZE}.pt (YOLO11s)")

# Train the model
print(f"\n🚀 Starting training...")
print(f"   Model: YOLO11s")
print(f"   Dataset: {DATA_YAML}")
print(f"   Classes: {num_classes if isinstance(num_classes, int) else 'Auto-detected from data.yaml'}")
print(f"   Epochs: {EPOCHS}")
print(f"   Image Size: {IMG_SIZE}")
print(f"   Batch Size: {BATCH_SIZE}")

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project='runs/detect',
    name=PROJECT_NAME,
    save=True,
    plots=True,
    val=True,
    device=device,
    verbose=True
)

print(f"\n✅ Training completed!")
print(f"   Results saved to: {results.save_dir}")

## 6. View Training Results

In [ ]:
# Display training results
from IPython.display import Image, display
import os

results_dir = results.save_dir

# Show results image
results_img = os.path.join(results_dir, 'results.png')
if os.path.exists(results_img):
    display(Image(results_img))
    print("📊 Training Results Plot")

# Show confusion matrix
confusion_matrix = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(confusion_matrix):
    display(Image(confusion_matrix))
    print("📊 Confusion Matrix")

# Print metrics
print("\n📈 Training Metrics:")
print(f"   Best mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"   Best mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"   Best Precision: {results.results_dict.get('metrics/precision(B)', 'N/A')}")
print(f"   Best Recall: {results.results_dict.get('metrics/recall(B)', 'N/A')}")

## 7. Save Model Weights

In [ ]:
# Copy best model to models directory
import shutil
from pathlib import Path

# Create models directory
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)

# Copy best model
best_model_src = Path(results_dir) / 'weights' / 'best.pt'
best_model_dst = models_dir / f'{PROJECT_NAME}-best.pt'

if best_model_src.exists():
    shutil.copy(best_model_src, best_model_dst)
    print(f"✅ Best model saved to: {best_model_dst}")
    print(f"   File size: {best_model_dst.stat().st_size / (1024*1024):.2f} MB")
else:
    print(f"⚠️ Best model not found at: {best_model_src}")

# Copy last checkpoint (optional)
last_model_src = Path(results_dir) / 'weights' / 'last.pt'
last_model_dst = models_dir / f'{PROJECT_NAME}-last.pt'

if last_model_src.exists():
    shutil.copy(last_model_src, last_model_dst)
    print(f"✅ Last checkpoint saved to: {last_model_dst}")

## 8. Optional: Save to Google Drive

If you mounted Google Drive, you can save models there for backup.

In [ ]:
# Optional: Save to Google Drive
# Uncomment and modify the path as needed

# drive_backup_path = f'/content/drive/MyDrive/HooperAI/models/{PROJECT_NAME}'
# os.makedirs(drive_backup_path, exist_ok=True)
# 
# if best_model_dst.exists():
#     shutil.copy(best_model_dst, os.path.join(drive_backup_path, f'{PROJECT_NAME}-best.pt'))
#     print(f"✅ Model backed up to Google Drive: {drive_backup_path}")

## 9. Test Model Inference (Optional)

Test the trained model on a sample image.

In [ ]:
# Load trained model and run inference
if best_model_dst.exists():
    # Load the best model
    trained_model = YOLO(str(best_model_dst))
    
    # Find a sample image from validation set
    val_images_dir = os.path.join(val_path, 'images')
    if os.path.exists(val_images_dir):
        sample_images = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.png'))]
        if sample_images:
            sample_image = os.path.join(val_images_dir, sample_images[0])
            
            # Run inference
            results = trained_model(sample_image)
            
            # Display results
            for r in results:
                im_array = r.plot()  # plot a BGR numpy array of predictions
                display(Image.fromarray(im_array[..., ::-1]))  # RGB PIL image
                print(f"✅ Inference on: {sample_images[0]}")
                print(f"   Detections: {len(r.boxes)}")
        else:
            print("⚠️ No sample images found in validation set")
    else:
        print("⚠️ Validation images directory not found")
else:
    print("⚠️ Model file not found for inference")